# Assignment 3 (bonus) — Walking the latent space

**CSCI 4052U — Machine Learning II** · **20 bonus marks**

The basic assignment swept the latent plane on a fixed grid. This one asks where the
data actually sits in that plane, and then takes a walk through it.

| | |
|---|---|
| 1 | Encode the test set and plot the ten digit classes on the latent plane |
| 2 | Find each class centroid, and lay a 50-point path through them in order |
| 3 | Decode every point of the path |
| 4 | Write the sequence out as an animated GIF |

**This notebook reads `vae.pt`**, written by the last cell of
`assignment3.ipynb`. Run that notebook to completion first and keep this one in the
same directory. Nothing is trained here.

In [ ]:
# COLAB SETUP -- run this cell first. Outside Colab it does nothing.
import os
import subprocess
import sys
import urllib.request

try:  # find_spec raises when the parent package is absent, so import instead
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab ships torch and torchvision; install only what it lacks.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightning", "pandas", "imageio"], check=True)


def _fetch(name, base="https://raw.githubusercontent.com/kenpu-uoit/csci4052u-machine-learning-2/main/assignments/assignment3"):
    """Download a support file when it is not already beside the notebook."""
    if not os.path.exists(name):
        urllib.request.urlretrieve(f"{base}/{name}", name)
        print(f"downloaded {name}")


if IN_COLAB:
    _fetch("components.py")
    _fetch("utils.py")
    _fetch("train.py")
    _fetch("z_samples.pt")

print("setup complete")

In [ ]:
import numpy as np
import torch
import torchvision
from torchvision import transforms

from components import D_Z
from train import load_trained, encode_dataset, decode_z
from utils import show_grid, show_scatter, to_frames
from utils import check_latents, check_path, check_gif

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

test_set = torchvision.datasets.MNIST(root="./data", train=False,
                                      download=True, transform=transforms.ToTensor())

# The model trained in the basic assignment. Nothing is trained in this notebook.
vae = load_trained("vae.pt", device=DEVICE)
print(f"{len(test_set)} test images | device {DEVICE}")

---

# B1 — Encode the test set and plot it (6 marks)

`encode_dataset(vae, dataset)` returns `(mu, labels)`: one 2-D point per image, keeping
only the **mean** of $q(z|x,\theta)$, and the digit label beside it.

1. Call it on `test_set`.
2. Plot with `show_scatter(test_mu, test_labels, annotate_centroids=True, title=...)`.

`annotate_centroids=True` marks each class centre with its digit, which is what makes
B2 readable.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

# test_mu, test_labels = ...
# show_scatter(...)

In [ ]:
# ----- CHECK B1 ---------------------------------------------------------------
check_latents(test_mu, test_labels, n_expected=len(test_set))

---

# B2 — Centroids, and a path through them (6 marks)

**The centroids.** For each digit $d = 0, 1, \ldots, 9$, average the encoded means of
every test image with that label:

$$c_d = \frac{1}{|\{i : y_i = d\}|} \sum_{i : y_i = d} \mu_i$$

Build `centroids`, shaped `(10, 2)`, with row $d$ holding $c_d$.

**The path.** Lay **exactly 50 points** along the polyline
$c_0 \to c_1 \to \cdots \to c_9$. Spacing them evenly *along the path* rather than
evenly per segment keeps the animation at a constant speed, and `np.interp` does it in
four lines:

```python
seg = np.linalg.norm(np.diff(c, axis=0), axis=1)   # length of each of the 9 segments
cum = np.concatenate([[0.0], np.cumsum(seg)])      # distance to each centroid
t   = np.linspace(0.0, cum[-1], 50)                # 50 equally spaced distances
path = np.stack([np.interp(t, cum, c[:, 0]),
                 np.interp(t, cum, c[:, 1])], axis=1)
```

Return `path` as a **torch tensor** of shape `(50, 2)`, then plot it over the scatter
with `show_scatter(..., extra=path, extra_label="the walk")`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

# centroids = ...   # (10, 2)
# path = ...        # (50, 2)
# show_scatter(...)

In [ ]:
# ----- CHECK B2 ---------------------------------------------------------------
check_path(path, centroids=centroids, n_expected=50)

---

# B3 — Decode the path (4 marks)

`decode_z(vae, path)` turns the 50 latent points into 50 images, `(50, 1, 28, 28)`.

Then show the first 25 as a 5×5 grid, so the morph is visible as stills before it
becomes an animation:

```python
show_grid(frames_t[:25], nrow=5, figsize=(6, 6), suptitle=...)
```

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

# walk_images = ...
# show_grid(...)

In [ ]:
# ----- CHECK B3 ---------------------------------------------------------------
assert tuple(walk_images.shape) == (50, 1, 28, 28), \
    f"expected (50, 1, 28, 28), got {tuple(walk_images.shape)}"
print(f"[ok] 50 images decoded along the path, "
      f"pixels in [{walk_images.min():.2f}, {walk_images.max():.2f}]")

---

# B4 — Write the animation (4 marks)

`to_frames(images)` converts the float images to the `uint8` array `imageio` wants, and
magnifies them ×4 so the result is not a postage stamp. Then write the GIF with
**`imageio.v3`**:

```python
import imageio.v3 as iio

frames = to_frames(walk_images)          # (50, 112, 112) uint8
iio.imwrite("latent_walk.gif", frames, duration=100, loop=0)
```

`duration` is milliseconds per frame and `loop=0` repeats forever. Call the file
**`latent_walk.gif`** — the check looks for that name.

Display it inline afterwards so the output survives in the submitted notebook:

```python
from IPython.display import Image
Image(filename="latent_walk.gif")
```

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

# frames = ...
# iio.imwrite("latent_walk.gif", ...)

In [ ]:
# ----- CHECK B4 ---------------------------------------------------------------
check_gif("latent_walk.gif", n_expected=50)

### **Your answer** — what the walk shows (included in B4's marks)

In two or three sentences: the animation never shows a sharp digit halfway between two
centroids. Say what it shows instead, and what that tells you about the region of the
latent plane lying between two classes.

> *Replace this line with your answer.*

---

## Marking

| | | Marks |
|---|---|---|
| B1 | the test set encoded and scattered by class | 6 |
| B2 | the ten centroids, and 50 points evenly spaced along the path | 6 |
| B3 | the path decoded to 50 images, shown as stills | 4 |
| B4 | `latent_walk.gif`, 50 frames, plus the answer cell | 4 |
| | **Total** | **20 bonus** |

The GIF must be in the submitted directory alongside the notebook. A notebook whose
animation cell shows no output is marked on its code alone.